In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_er_dataset(num_records=1000):
    np.random.seed(42)
    random.seed(42)

    snapshot_date = datetime(2026, 7, 6, 12, 0, 0)

    # Advanced Complaint Profiles with Temporal Weights
    # Weights: (Morning 6-12, Afternoon 12-18, Evening 18-0, Night 0-6)
    # DayWeights: (Weekday, Weekend)
    complaint_profiles = {
        # -- Cardiology pool --
        "Chest Pain": {
            "stats": (2, 0.4, 0.4), # triage, crit_prob, adm_boost
            "time_weights": [0.4, 0.2, 0.2, 0.2], # Early morning peak
            "day_weights": [1.0, 1.0]
        },
        "Palpitations": {
            "stats": (3, 0.15, 0.15),
            "time_weights": [0.3, 0.3, 0.2, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Shortness of Breath": {
            "stats": (2, 0.35, 0.3),
            "time_weights": [0.3, 0.2, 0.3, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Syncope": {
            "stats": (2, 0.4, 0.35),
            "time_weights": [0.3, 0.3, 0.2, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Hypertension": {
            "stats": (4, 0.05, 0.1),
            "time_weights": [0.3, 0.3, 0.2, 0.2],
            "day_weights": [1.1, 0.9]
        },

        # -- Neurology pool --
        "Stroke Symptoms": {
            "stats": (1, 0.7, 0.6),
            "time_weights": [0.5, 0.2, 0.1, 0.2], # Morning peak
            "day_weights": [1.0, 1.0]
        },
        "Seizure": {
            "stats": (2, 0.5, 0.4),
            "time_weights": [0.2, 0.2, 0.3, 0.3],
            "day_weights": [1.0, 1.0]
        },
        "Headache": {
            "stats": (4, 0.05, 0.05),
            "time_weights": [0.2, 0.3, 0.3, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Confusion": {
            "stats": (3, 0.3, 0.3),
            "time_weights": [0.2, 0.2, 0.3, 0.3],
            "day_weights": [1.0, 1.0]
        },
        "Loss of Consciousness": {
            "stats": (1, 0.6, 0.5),
            "time_weights": [0.2, 0.2, 0.3, 0.3],
            "day_weights": [1.0, 1.0]
        },

        # -- Orthopedics pool --
        "Trauma": {
            "stats": (2, 0.5, 0.3),
            "time_weights": [0.1, 0.3, 0.4, 0.2], # Evening/Night
            "day_weights": [0.7, 1.3] # Weekend peak
        },
        "Fracture": {
            "stats": (3, 0.05, 0.1),
            "time_weights": [0.1, 0.4, 0.4, 0.1], # Active hours
            "day_weights": [0.5, 1.5]
        },
        "Back Pain": {
            "stats": (4, 0.01, 0.05),
            "time_weights": [0.3, 0.4, 0.2, 0.1], # Day hours
            "day_weights": [1.2, 0.8] # Weekday work-related
        },
        "Joint Injury": {
            "stats": (4, 0.05, 0.05),
            "time_weights": [0.1, 0.4, 0.4, 0.1],
            "day_weights": [0.6, 1.4]
        },
        "Dislocation": {
            "stats": (3, 0.1, 0.15),
            "time_weights": [0.1, 0.3, 0.4, 0.2],
            "day_weights": [0.5, 1.5]
        },

        # -- Pediatrics pool --
        "Fever": {
            "stats": (4, 0.05, 0.1),
            "time_weights": [0.1, 0.2, 0.5, 0.2], # Evening peak (parents coming home)
            "day_weights": [0.8, 1.2]
        },
        "Vomiting": {
            "stats": (4, 0.05, 0.1),
            "time_weights": [0.1, 0.3, 0.4, 0.2],
            "day_weights": [0.9, 1.1]
        },
        "Abdominal Pain": {
            "stats": (3, 0.1, 0.2),
            "time_weights": [0.2, 0.3, 0.3, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Respiratory Infection": {
            "stats": (3, 0.15, 0.2),
            "time_weights": [0.1, 0.2, 0.4, 0.3],
            "day_weights": [0.9, 1.1]
        },
        "Allergic Reaction": {
            "stats": (3, 0.2, 0.1),
            "time_weights": [0.1, 0.4, 0.4, 0.1],
            "day_weights": [1.0, 1.0]
        },

        # -- General pool (General ER, General Practice, Gastroenterology, Renal) --
        "Difficulty Breathing": {
            "stats": (2, 0.4, 0.4),
            "time_weights": [0.3, 0.2, 0.3, 0.2],
            "day_weights": [1.0, 1.0]
        },
        "Head Injury": {
            "stats": (2, 0.3, 0.3),
            "time_weights": [0.1, 0.2, 0.4, 0.3], # Night/Evening
            "day_weights": [0.6, 1.4]
        },
        "Laceration": {
            "stats": (4, 0.05, 0.05),
            "time_weights": [0.1, 0.3, 0.4, 0.2],
            "day_weights": [0.5, 1.5] # Weekend DIY/Sports
        },
        "Mental Health Crisis": {
            "stats": (3, 0.1, 0.3),
            "time_weights": [0.1, 0.2, 0.3, 0.4], # Late night
            "day_weights": [1.0, 1.0]
        }
    }

    # Department-specific complaint pools: a chief_complaint can ONLY be
    # generated for the department it belongs to.
    department_pools = {
        "Cardiology": ["Chest Pain", "Palpitations", "Shortness of Breath", "Syncope", "Hypertension"],
        "Neurology": ["Stroke Symptoms", "Seizure", "Headache", "Confusion", "Loss of Consciousness"],
        "Orthopedics": ["Fracture", "Back Pain", "Joint Injury", "Trauma", "Dislocation"],
        "Pediatrics": ["Fever", "Vomiting", "Abdominal Pain", "Respiratory Infection", "Allergic Reaction"],
    }
    general_pool = ["Difficulty Breathing", "Head Injury", "Laceration", "Mental Health Crisis"]
    general_departments = ["General ER", "General Practice", "Gastroenterology", "Renal"]

    # Relative frequency of each non-pediatric department (mirrors the
    # original dataset's department distribution)
    dept_weights = {
        "Cardiology": 81, "Orthopedics": 67, "General ER": 63,
        "Gastroenterology": 55, "Renal": 51, "Neurology": 47, "General Practice": 45
    }
    dept_names = list(dept_weights.keys())
    dept_w = list(dept_weights.values())

    def pick_complaint(pool, hour, is_weekend):
        if 6 <= hour < 12: slot = 0
        elif 12 <= hour < 18: slot = 1
        elif 18 <= hour <= 23: slot = 2
        else: slot = 3

        probs = []
        for c in pool:
            tw = complaint_profiles[c]["time_weights"][slot]
            dw = complaint_profiles[c]["day_weights"][1 if is_weekend else 0]
            probs.append(tw * dw)
        total_p = sum(probs)
        probs = [p / total_p for p in probs]
        return random.choices(pool, weights=probs)[0]

    # Generate records
    data = []
    patient_ids = [f"P{str(i).zfill(6)}" for i in range(1, num_records + 1)]

    # We will generate arrival times first, then assign complaints based on probabilities for that time
    arrival_times = []
    for _ in range(num_records):
        r = random.random()
        if r < 0.70: # Historical
            days_back = random.randint(2, 180)
        elif r < 0.90: # Recent
            days_back = random.uniform(0.1, 2)
        else: # Live
            days_back = random.uniform(0, 0.1)

        arr_time = snapshot_date - timedelta(days=days_back, hours=random.randint(0,23), minutes=random.randint(0,59))
        arrival_times.append(arr_time)

    arrival_times.sort()

    for i in range(num_records):
        arr_time = arrival_times[i]
        hour = arr_time.hour
        is_weekend = arr_time.weekday() >= 5

        # Demographics (age decided before department/complaint, since
        # Pediatrics is strictly age-gated)
        age = random.randint(0, 100)
        gender = random.choices(["Male", "Female", "Other"], weights=[48, 48, 4])[0]
        race = random.choices(["White", "Black", "Hispanic", "Asian", "Other"], weights=[60, 15, 15, 7, 3])[0]

        # Department -> Complaint (complaint can only come from its own
        # department's pool)
        if age < 18:
            dept = "Pediatrics"
            complaint = pick_complaint(department_pools["Pediatrics"], hour, is_weekend)
        else:
            dept = random.choices(dept_names, weights=dept_w)[0]
            if dept in department_pools:
                complaint = pick_complaint(department_pools[dept], hour, is_weekend)
                # A handful of cardiology/neurology complaints are strongly
                # associated with older patients
                if complaint in ["Chest Pain", "Stroke Symptoms", "Hypertension", "Syncope", "Loss of Consciousness"]:
                    age = random.randint(50, 95)
            else:
                complaint = pick_complaint(general_pool, hour, is_weekend)
                if complaint == "Fever" and random.random() < 0.6:
                    age = random.randint(18, 40)

        profile = complaint_profiles[complaint]
        stats = profile["stats"]

        # Mode and Criticality
        mode = random.choices(["Walk-in", "Ambulance", "Referral"], weights=[60, 25, 15])[0]
        crit_prob = stats[1]
        if mode == "Ambulance": crit_prob += 0.3
        is_critical = random.random() < min(0.95, crit_prob)

        # Triage
        if is_critical:
            triage = random.randint(1, 2)
        else:
            triage = max(1, min(5, int(np.random.normal(stats[0], 0.7))))

        # Wait time
        base_wait = {1: 2, 2: 15, 3: 45, 4: 90, 5: 120}[triage]
        # Congestion factor: more patients in the last 4 hours increases wait
        recent_count = sum(1 for t in arrival_times[:i] if arr_time - t < timedelta(hours=4))
        congestion_mult = 1 + (recent_count / 50)
        wait_time = max(0, int(np.random.normal(base_wait * congestion_mult, base_wait * 0.2)))

        # Admission and Disposition
        adm_prob = 0.1 + stats[2]
        if age > 70: adm_prob += 0.2
        if is_critical: adm_prob += 0.3
        admitted = random.random() < min(0.95, adm_prob)

        if admitted:
            disposition = random.choices(["Admitted", "Transferred"], weights=[90, 10])[0]
        else:
            disposition = random.choices(["Discharged", "Expired", "Left Without Being Seen"], weights=[95, 1, 4])[0]
            if is_critical and disposition == "Discharged" and random.random() < 0.2:
                disposition = "Expired"

        # LOS
        if admitted: base_los = random.randint(300, 1800)
        else: base_los = {1: 360, 2: 300, 3: 240, 4: 180, 5: 120}[triage]

        if disposition == "Left Without Being Seen":
            los = random.randint(20, max(21, wait_time))
        else:
            los = max(wait_time + 20, int(np.random.normal(base_los, base_los * 0.25)))

        # Case Management
        cm_prob = 0.05
        if age > 75: cm_prob += 0.3
        if is_critical: cm_prob += 0.2
        case_management = random.random() < min(0.8, cm_prob)

        # Status
        time_since_arrival = snapshot_date - arr_time
        hours_since_arrival = time_since_arrival.total_seconds() / 3600
        if hours_since_arrival > (los / 60):
            status = "Admitted" if admitted else "Discharged"
        elif hours_since_arrival < (wait_time / 60):
            status = "Waiting"
        else:
            status = "Under Treatment"

        bed_id = None
        if status in ["Admitted", "Under Treatment"]:
            prefix = dept[:3].upper()
            bed_id = f"{prefix}-{random.randint(1, 50):02d}"

        data.append({
            "patient_id": patient_ids[i],
            "arrival_time": arr_time.strftime("%Y-%m-%d %H:%M:%S"),
            "age": age,
            "gender": gender,
            "race": race,
            "department": dept,
            "arrival_mode": mode,
            "triage_level": triage,
            "wait_time_min": wait_time,
            "admitted": admitted,
            "is_critical": is_critical,
            "case_management": case_management,
            "current_status": status,
            "bed_id": bed_id,
            "length_of_stay_min": los,
            "disposition": disposition,
            "chief_complaint": complaint
        })

    df = pd.DataFrame(data)
    ordered_cols = [
        "patient_id", "arrival_time", "age", "gender", "race", "department",
        "arrival_mode", "triage_level", "wait_time_min", "admitted",
        "is_critical", "case_management", "current_status", "bed_id",
        "length_of_stay_min", "disposition", "chief_complaint"
    ]
    return df[ordered_cols]

if __name__ == "__main__":
    df = generate_er_dataset(500)
    df.to_csv("er_dataset_v5.csv", index=False)
    print("Creative Dataset v5 (department-scoped complaints) generated successfully.")

Creative Dataset v5 (department-scoped complaints) generated successfully.


In [2]:
from google.colab import files
files.download('er_dataset_v5.csv')